In [3]:
import sys
print(sys.executable)

c:\Users\user\anaconda3\python.exe


In [4]:

import pandas as pd

df = pd.read_csv("credit_card_default_clean.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())


Dataset shape: (30000, 25)

Column names:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']


In [8]:
df = df.rename(columns={
    "default payment next month": "default_next_month"
})

In [5]:

from sqlalchemy import create_engine, text
from getpass import getpass
from urllib.parse import quote_plus

# Enter your MySQL password securely
password = getpass("MySQL password: ")

engine = create_engine(
    f"mysql+pymysql://root:{quote_plus(password)}"
    "@localhost:3306/credit_card_default_db"
)

# Test connection
with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE()"))
    print("Connected to:", result.scalar())


Connected to: credit_card_default_db


In [9]:

from sqlalchemy import text

# Check required columns
required = (
    ["ID", "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE",
     "AGE", "default_next_month"]
    + ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
    + [f"BILL_AMT{i}" for i in range(1, 7)]
    + [f"PAY_AMT{i}" for i in range(1, 7)]
)

missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing CSV columns: {missing}")

if df["ID"].isna().any() or df["ID"].duplicated().any():
    raise ValueError("Customer IDs must be unique and non-null.")

# Table 1: Customers
customers = df[
    ["ID", "LIMIT_BAL", "SEX", "EDUCATION",
     "MARRIAGE", "AGE", "default_next_month"]
].copy()

customers.columns = [
    "customer_id", "limit_bal", "sex", "education",
    "marriage", "age", "default_next_month"
]

# Helper: Convert six monthly columns into long format
def make_history(columns, value_name):
    result = df[["ID"] + columns].melt(
        id_vars="ID",
        value_vars=columns,
        var_name="source_month",
        value_name=value_name
    )

    month_mapping = {
        column: i for i, column in enumerate(columns, start=1)
    }

    result["month_index"] = result["source_month"].map(
        month_mapping
    )

    return result[
        ["ID", "month_index", value_name]
    ].rename(columns={"ID": "customer_id"})

# Table 2: Repayment Status
repayment = make_history(
    ["PAY_0", "PAY_2", "PAY_3",
     "PAY_4", "PAY_5", "PAY_6"],
    "payment_status"
)

# Table 3: Billing History
billing = make_history(
    [f"BILL_AMT{i}" for i in range(1, 7)],
    "bill_amount"
)

# Table 4: Payment History
payments = make_history(
    [f"PAY_AMT{i}" for i in range(1, 7)],
    "payment_amount"
)

# Import data in one transaction
with engine.begin() as conn:

    tables = [
        "customers", "repayment_status",
        "billing_history", "payment_history"
    ]

    # Prevent accidental duplicate imports
    for table in tables:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).scalar()

        if count != 0:
            raise ValueError(
                f"Table {table} already contains data. "
                "Import stopped to prevent duplicates."
            )

    for table_name, data in [
        ("customers", customers),
        ("repayment_status", repayment),
        ("billing_history", billing),
        ("payment_history", payments)
    ]:
        data.to_sql(
            table_name,
            con=conn,
            if_exists="append",
            index=False,
            chunksize=2000
        )

        print(f"{table_name}: {len(data):,} rows imported")

print("\nImport completed successfully!")


customers: 30,000 rows imported
repayment_status: 180,000 rows imported
billing_history: 180,000 rows imported
payment_history: 180,000 rows imported

Import completed successfully!


In [10]:

from sqlalchemy import text

with engine.connect() as conn:
    for table in [
        "customers",
        "repayment_status",
        "billing_history",
        "payment_history"
    ]:
        count = conn.execute(
            text(f"SELECT COUNT(*) FROM {table}")
        ).scalar()

        print(f"{table}: {count:,} rows")


customers: 30,000 rows
repayment_status: 180,000 rows
billing_history: 180,000 rows
payment_history: 180,000 rows
